# Model Architecture Explorer
Instantiates every model in the project and prints its architecture + parameter count.
No checkpoint loading — all models use default/random weights.

In [12]:
import sys; sys.path.insert(0, "../src")
import torch
import torch.nn as nn
import torch.nn.functional as F
import zuko

from suplat.models.byol_models import (
    BYOLEncoder, ProjectionHead, PredictionHead,
    create_efficientnet_b0_backbone,
    BYOLEfficient, BYOLEfficientNetB0,
    PCAProjection,
)
from suplat.models.generative_models import FlowMatchingUNet

def param_count(m):
    return f"{sum(p.numel() for p in m.parameters()):,}"

def trainable_param_count(m):
    return f"{sum(p.numel() for p in m.parameters() if p.requires_grad):,}"

def online_param_count(m):
    """Count params in the online branch (encoder + projector + predictor)."""
    parts = [m.online_encoder, m.online_projector]
    if m.online_predictor is not None:
        parts.append(m.online_predictor)
    total = sum(p.numel() for part in parts for p in part.parameters())
    return f"{total:,}"

# ---------------------------------------------------------------------------
# BYOLFineTuner — inlined from scripts/train_finetuning.py
# ---------------------------------------------------------------------------
TRAINING_MODE_DESCRIPTIONS = {
    1: "Freeze embeddings: frozen encoder and projector; train linear classifier only.",
    2: "Freeze features: frozen encoder; fine-tune projector and train linear classifier.",
    3: "No freeze: fine-tune encoder and projector; train linear classifier.",
    4: "Supervised: initialize from scratch and train encoder, projector, and linear classifier.",
}

class BYOLFineTuner(nn.Module):
    def __init__(self, byol_model, num_classes=21, training_mode=3, dropout_rate=0.0):
        super().__init__()
        self.encoder       = byol_model.online_encoder
        self.projector     = byol_model.online_projector
        self.training_mode = training_mode
        self.dropout       = nn.Dropout(p=dropout_rate)
        self.classifier    = nn.Linear(128, num_classes)
        self.apply_training_mode(training_mode)

    def apply_training_mode(self, training_mode):
        for p in self.encoder.parameters():
            p.requires_grad = training_mode in {3, 4}
        for p in self.projector.parameters():
            p.requires_grad = training_mode in {2, 3, 4}
        for p in self.classifier.parameters():
            p.requires_grad = True

    def train(self, mode=True):
        super().train(mode)
        if mode:
            if self.training_mode in {1, 2}:
                self.encoder.eval()
            if self.training_mode == 1:
                self.projector.eval()
        return self

    def forward(self, x):
        z = self.encoder(x)
        z = self.projector(z)
        z = self.dropout(z)
        return self.classifier(z)

# ---------------------------------------------------------------------------
# ProjectionDecoder — inlined from scripts/train_generative.py
# ---------------------------------------------------------------------------
class ProjectionDecoder(nn.Module):
    """FC decoder: z → 89×89. Fast baseline, tends to produce blurry outputs."""
    def __init__(self, proj_dim=256, dropout=0.0):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(proj_dim, 1024), nn.BatchNorm1d(1024), nn.ReLU(inplace=True), nn.Dropout(p=dropout),
            nn.Linear(1024, 256 * 5 * 5), nn.BatchNorm1d(256 * 5 * 5), nn.ReLU(inplace=True), nn.Dropout(p=dropout),
        )
        def up(ic, oc):
            return nn.Sequential(
                nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
                nn.Conv2d(ic, oc, 3, padding=1), nn.BatchNorm2d(oc), nn.ReLU(inplace=True),
            )
        self.up1 = up(256, 128); self.up2 = up(128, 64)
        self.up3 = up(64,  32);  self.up4 = up(32,  16)
        self.out_conv = nn.Conv2d(16, 1, 3, padding=1)

    def forward(self, z):
        x = self.fc(z).view(-1, 256, 5, 5)
        x = self.up1(x); x = self.up2(x); x = self.up3(x); x = self.up4(x)
        x = F.interpolate(x, size=(89, 89), mode="bilinear", align_corners=False)
        return torch.sigmoid(self.out_conv(x))

print("Imports OK")

Imports OK


## 1. BYOL Encoder
Custom ResNet-style encoder for 89×89 greyscale images. Outputs a 512-dim representation.

In [13]:
enc = BYOLEncoder()
print(enc)
print(f"\nParams: {param_count(enc)}")

BYOLEncoder(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer1): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
  )
  (layer2): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inpla

## 2. BYOL Projection & Prediction Heads
Both are 2-layer MLPs with BN. Projector maps 512 → 256; predictor maps 256 → 256.

In [24]:
proj = ProjectionHead(in_dim=1280, hidden_dim=4096, out_dim=128)
pred = PredictionHead(in_dim=128, hidden_dim=4096, out_dim=128)

print("--- ProjectionHead ---")
print(proj)
print(f"Params: {param_count(proj)}")

print("\n--- PredictionHead ---")
print(pred)
print(f"Params: {param_count(pred)}")

--- ProjectionHead ---
ProjectionHead(
  (net): Sequential(
    (0): Linear(in_features=1280, out_features=4096, bias=True)
    (1): BatchNorm1d(4096, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Linear(in_features=4096, out_features=128, bias=True)
  )
)
Params: 5,779,584

--- PredictionHead ---
PredictionHead(
  (net): Sequential(
    (0): Linear(in_features=128, out_features=4096, bias=True)
    (1): BatchNorm1d(4096, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Linear(in_features=4096, out_features=128, bias=True)
  )
)
Params: 1,060,992


## 3. EfficientNet-B0 Backbone
Pretrained on ImageNet; first conv replaced to accept 1-channel input. Classifier head removed → 1280-dim output.

**Note:** instantiation downloads ImageNet weights (~20 MB) on first run.

In [15]:
backbone = create_efficientnet_b0_backbone(num_channels=1, dropout_rate=0.2)
print(backbone)
print(f"\nParams: {param_count(backbone)}")

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

## 4. Full BYOL Models: BYOLEfficient & BYOLEfficientNetB0
Both use `feature_compression_mode='mlp'` so the online predictor is constructed at init time.
The target network is a frozen EMA copy of the online branch and is excluded from the online param count.

In [16]:
byol_eff = BYOLEfficient(feature_compression_mode='mlp', projection_dim=128)

print("=== BYOLEfficient ===")
print("--- online_encoder ---"); print(byol_eff.online_encoder)
print("--- online_projector ---"); print(byol_eff.online_projector)
print("--- online_predictor ---"); print(byol_eff.online_predictor)
print(f"\nOnline-branch params: {online_param_count(byol_eff)}")
print(f"Total params (incl. frozen target): {param_count(byol_eff)}")

=== BYOLEfficient ===
--- online_encoder ---
BYOLEncoder(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer1): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
  )
  (layer2): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, 

In [17]:
byol_effnet = BYOLEfficientNetB0(feature_compression_mode='mlp', projection_dim=128)

print("=== BYOLEfficientNetB0 ===")
print("--- online_encoder ---"); print(byol_effnet.online_encoder)
print("--- online_projector ---"); print(byol_effnet.online_projector)
print("--- online_predictor ---"); print(byol_effnet.online_predictor)
print(f"\nOnline-branch params: {online_param_count(byol_effnet)}")
print(f"Total params (incl. frozen target): {param_count(byol_effnet)}")

=== BYOLEfficientNetB0 ===
--- online_encoder ---
EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigm

## 5. BYOLFineTuner — mode 2 (freeze encoder, train projector + classifier)
Wraps the online encoder + projector from a pretrained `BYOLEfficientNetB0` and adds a linear classification head.
Mode 2: EfficientNet-B0 encoder is **frozen** (weights fixed, kept in eval); projector and classifier are trained.
The classifier input dim is hardcoded to 128 (matching the projection dim of the training run).

In [18]:
_dummy_byol = BYOLEfficientNetB0(projection_dim=128, feature_compression_mode='mlp')
finetuner = BYOLFineTuner(_dummy_byol, num_classes=5, training_mode=2)

print(f"Encoder params (frozen):   {param_count(finetuner.encoder)}")
print(f"Projector params:          {param_count(finetuner.projector)}")
print(f"Classifier params:         {param_count(finetuner.classifier)}")
print(f"Total params:              {param_count(finetuner)}")
print(f"Trainable params (mode 2): {trainable_param_count(finetuner)}")

Encoder params (frozen):   4,006,972
Projector params:          5,779,584
Classifier params:         645
Total params:              9,787,201
Trainable params (mode 2): 5,780,229


## 6. Logistic Regression on BYOL Features
The primary classifier used in evaluation. No neural architecture to print.

In [19]:
from sklearn.linear_model import LogisticRegression
import numpy as np

lr = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', multi_class='ovr')
print(lr)
print()
print("Usage:")
print("  features = byol_model.online_encoder(images).detach().numpy()  # (N, 512)")
print("  projections = byol_model.online_projector(features).numpy()     # (N, proj_dim)")
print("  lr.fit(projections[train_idx], labels[train_idx])")
print("  preds = lr.predict(projections[test_idx])")
print()
print("No learnable parameters — scikit-learn fits coefficients analytically/iteratively.")

LogisticRegression(max_iter=1000, multi_class='ovr')

Usage:
  features = byol_model.online_encoder(images).detach().numpy()  # (N, 512)
  projections = byol_model.online_projector(features).numpy()     # (N, proj_dim)
  lr.fit(projections[train_idx], labels[train_idx])
  preds = lr.predict(projections[test_idx])

No learnable parameters — scikit-learn fits coefficients analytically/iteratively.


## 7. Flow-Matching U-Net Decoder
Velocity network `v_θ(x_t, t, z)` conditioned on BYOL projection `z` and scalar timestep `t`.
U-Net with sinusoidal time embedding + AdaIN-style residual blocks.

In [20]:
unet = FlowMatchingUNet(z_dim=128, t_dim=128, base_ch=32)
print(unet)
print(f"\nParams: {param_count(unet)}")

FlowMatchingUNet(
  (t_embed): Sequential(
    (0): _SinusoidalEmbedding()
    (1): Linear(in_features=128, out_features=128, bias=True)
    (2): SiLU()
    (3): Linear(in_features=128, out_features=128, bias=True)
  )
  (in_conv): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (res_e1): _CondResBlock(
    (norm1): GroupNorm(8, 32, eps=1e-05, affine=True)
    (norm2): GroupNorm(8, 32, eps=1e-05, affine=True)
    (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (cond): Linear(in_features=256, out_features=64, bias=True)
  )
  (down1): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (res_e2): _CondResBlock(
    (norm1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (norm2): GroupNorm(8, 64, eps=1e-05, affine=True)
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), str

## 8. Zuko Neural Spline Flow (NSF)
Conditional normalising flow used to model the distribution of BYOL projections given class labels.
8 rational-quadratic spline transforms with random permutation between layers.

In [21]:
nsf = zuko.flows.NSF(features=128, context=10, transforms=8, randperm=True)
print(nsf)
print(f"\nParams: {param_count(nsf)}")

NSF(
  (transform): LazyComposedTransform(
    (0): MaskedAutoregressiveTransform(
      (base): MonotonicRQSTransform(bins=8)
      (order): [57, 122, 106, 7, 54, ..., 39, 1, 93, 32, 62]
      (hyper): MaskedMLP(
        (0): MaskedLinear(in_features=138, out_features=64, bias=True)
        (1): ReLU()
        (2): MaskedLinear(in_features=64, out_features=64, bias=True)
        (3): ReLU()
        (4): MaskedLinear(in_features=64, out_features=2944, bias=True)
      )
    )
    (1): MaskedAutoregressiveTransform(
      (base): MonotonicRQSTransform(bins=8)
      (order): [106, 115, 126, 53, 81, ..., 14, 109, 41, 59, 68]
      (hyper): MaskedMLP(
        (0): MaskedLinear(in_features=138, out_features=64, bias=True)
        (1): ReLU()
        (2): MaskedLinear(in_features=64, out_features=64, bias=True)
        (3): ReLU()
        (4): MaskedLinear(in_features=64, out_features=2944, bias=True)
      )
    )
    (2): MaskedAutoregressiveTransform(
      (base): MonotonicRQSTransform(b

## 11. ProjectionDecoder (FC generative baseline)
Fully-connected decoder mapping a BYOL projection `z` back to an 89×89 image via 4 bilinear upsample blocks.
Faster to train than FlowMatchingUNet but produces blurrier outputs.

In [22]:
dec = ProjectionDecoder(proj_dim=128)
print(dec)
print(f"\nParams: {param_count(dec)}")

ProjectionDecoder(
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=1024, bias=True)
    (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.0, inplace=False)
    (4): Linear(in_features=1024, out_features=6400, bias=True)
    (5): BatchNorm1d(6400, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): Dropout(p=0.0, inplace=False)
  )
  (up1): Sequential(
    (0): Upsample(scale_factor=2.0, mode='bilinear')
    (1): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU(inplace=True)
  )
  (up2): Sequential(
    (0): Upsample(scale_factor=2.0, mode='bilinear')
    (1): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  